### Notebook 4 — Écriture Parquet (et chargement PostgreSQL bonus)

In [19]:
from pyspark.sql import SparkSession

In [20]:
spark = SparkSession.builder.appName("TradeCorp ETL notebook-4").getOrCreate()
spark

26/11/19 15:55:28 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


##### Q33 — Relire le Parquet
*Relire le fichier Parquet et vérifier que le nombre de lignes est identique à l'original. Afficher le schema — observer que les types sont préservés.*

In [21]:
# Lecture des fichiers Parquet
PATH = "/home/jovyan/tradecorp/data/gold"

df_parquet = spark.read.parquet(f"{PATH}/orders_enriched.parquet")
#Vérification de nombres de lignes
print(f"Nombres de lignes: {df_parquet.count()}")
#Print Schéma
print("Schéma du parquet")
df_parquet.printSchema()


Nombres de lignes: 893
Schéma du parquet
root
 |-- product_id: integer (nullable = true)
 |-- shipper_id: integer (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- prix_unitaire: double (nullable = true)
 |-- quantite: integer (nullable = true)
 |-- discount: double (nullable = true)
 |-- sous_total: double (nullable = true)
 |-- order_date: date (nullable = true)
 |-- required_date: date (nullable = true)
 |-- shipped_date: date (nullable = true)
 |-- freight: double (nullable = true)
 |-- ship_name: string (nullable = true)
 |-- ship_address: string (nullable = true)
 |-- ship_city: string (nullable = true)
 |-- ship_region: string (nullable = true)
 |-- ship_postal_code: string (nullable = true)
 |-- ship_country: string (nullable = true)
 |-- is_shipped: boolean (nullable = true)
 |-- company_name: string (nullable = true)
 |-- contact_name: string (nullable = true)
 |-- contact_tit

#### Q34 — Comparer CSV vs Parquet
*Comparer la taille du fichier CSV original vs le fichier Parquet. Observer le gain de compression. Expliquer
pourquoi Parquet est plus efficace.*

In [22]:
PATH = "/home/jovyan/tradecorp/data/gold"

df_csv = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{PATH}/orders_enriched.csv")
df_csv.show(5, truncate=False)


+----------+----------+-----------+-----------+--------+-------------+--------+--------+----------+----------+-------------+------------+-------+----------------------------+--------------------+---------+-----------+----------------+------------+----------+----------------------------+----------------+------------------+--------------------+-------------+--------+-----------+----------------+--------------+-----------+----------+---------+--------------------+----------+-------------+----------------+----------------+----------------+--------------+-----------+--------------------------------+-----------+-----------------+------------------+--------------+--------------+-------------+------------+--------+-------------+----------------------------------------------------------+-------+
|product_id|shipper_id|employee_id|customer_id|order_id|prix_unitaire|quantite|discount|sous_total|order_date|required_date|shipped_date|freight|ship_name                   |ship_address        |ship_ci

#### Comparer parquet et csv 

In [23]:
import os

PATH = "/home/jovyan/tradecorp/data/gold"

def get_size(path):
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total

parquet_size = get_size(f"{PATH}/orders_enriched.parquet")
csv_size = get_size(f"{PATH}/orders_enriched.csv")

print(f"Parquet size: {parquet_size / 1024:.2f} KB")
print(f"CSV size: {csv_size / 1024:.2f} KB")
print(f"CSV / Parquet ratio: {csv_size / parquet_size:.2f}x")

Parquet size: 67.71 KB
CSV size: 411.26 KB
CSV / Parquet ratio: 6.07x


In [24]:
# Conclusion
print("csv est 6.07x plus lourd que parquet")

csv est 6.07x plus lourd que parquet


#### Q35 — Partitionnement
*Écrire le DataFrame partitionné par country avec partitionBy('country'). Observer la structure des dossiers créés.*

In [25]:
import os
print(os.access("/home/jovyan/data/tmp", os.W_OK))

False


In [26]:
import os
PATH_output_partition = "/home/jovyan/data/tmp"

#df_full.write.mode("overwrite").parquet(f"{PATH}/orders_enriched.parquet")

df_parquet.write \
.mode("overwrite") \
.partitionBy("ship_country") \
.parquet("/tmp/partition_output")

# Affichage de la structure des dossiers créés

#### Q36 — Chargement PostgreSQL via JDBC
*Ajouter PostgreSQL au docker-compose. Écrire df_orders_enriched dans PostgreSQL via JDBC dans une table
orders_enriched.*

In [27]:
# 2. Ajouter le driver JDBC PostgreSQL à Spark

spark = (SparkSession.builder.appName("Orders Pipeline").config("spark.jars.packages", "org.postgresql:postgresql:42.7.4").getOrCreate())
spark.sparkContext.getConf().get("spark.jars.packages")

26/11/19 15:55:31 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [28]:
SparkSession.builder.config("spark.jars.packages", "org.postgresql:postgresql:42.7.3").getOrCreate()

26/11/19 15:55:31 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [ ]:
import os

jdbc_url = os.getenv(
    "POSTGRES_JDBC_URL",
    "jdbc:postgresql://airflow-postgres-tradecorp:5432/airflow",
)
postgres_user = os.getenv("POSTGRES_USER", "airflow")
postgres_password = os.getenv("POSTGRES_PASSWORD", "airflow")

(
    df_parquet.write
        .format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", "orders_enriched")
        .option("user", postgres_user)
        .option("password", postgres_password)
        .option("driver", "org.postgresql.Driver")
        .mode("overwrite")
        .save()
)


Py4JJavaError: An error occurred while calling o250.save.
: java.lang.ClassNotFoundException: org.postgresql.Driver
	at java.base/java.net.URLClassLoader.findClass(URLClassLoader.java:445)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:593)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:526)
	at org.apache.spark.sql.execution.datasources.jdbc.DriverRegistry$.register(DriverRegistry.scala:46)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$1(JDBCOptions.scala:103)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$1$adapted(JDBCOptions.scala:103)
	at scala.Option.foreach(Option.scala:407)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.<init>(JDBCOptions.scala:103)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcOptionsInWrite.<init>(JDBCOptions.scala:246)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcOptionsInWrite.<init>(JDBCOptions.scala:250)
	at or

#### Q37 — Vérifier dans pgAdmin
*Ouvrir pgAdmin, vérifier que la table orders_enriched existe et contient les données. Exécuter une requête SQL*
de validation.

In [ ]:
import os

jdbc_url = os.getenv(
    "POSTGRES_JDBC_URL",
    "jdbc:postgresql://airflow-postgres-tradecorp:5432/airflow",
)
postgres_user = os.getenv("POSTGRES_USER", "airflow")
postgres_password = os.getenv("POSTGRES_PASSWORD", "airflow")

df_check = (
    spark.read
        .format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", "orders_enriched")
        .option("user", postgres_user)
        .option("password", postgres_password)
        .option("driver", "org.postgresql.Driver")
        .load()
)

print(f"Données PostgreSQL : {df_check.count()} lignes")
df_check.show(5, truncate=False)


Py4JJavaError: An error occurred while calling o135.load.
: java.lang.ClassNotFoundException: org.postgresql.Driver
	at java.base/java.net.URLClassLoader.findClass(URLClassLoader.java:445)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:593)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:526)
	at org.apache.spark.sql.execution.datasources.jdbc.DriverRegistry$.register(DriverRegistry.scala:46)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$1(JDBCOptions.scala:103)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$1$adapted(JDBCOptions.scala:103)
	at scala.Option.foreach(Option.scala:407)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.<init>(JDBCOptions.scala:103)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.<init>(JDBCOptions.scala:41)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcRelationProvider.createRelation(JdbcRelationProvider.scala: